# Genixcure — PacBio PLA_6_13 HiFi BAM exact unique read counts

원본: [Genixcure Google Drive](https://drive.google.com/drive/folders/1vv7WLOiYTZ0FGXPmFciIN6d9sM6Rhg58)

- PacBio HiFi read는 이미 consensus read입니다. **Illumina paired-end `fastp` merging을 하지 않습니다.**
- 단일 HiFi read의 full-length nucleotide sequence를 기준으로 **100% exact-match dereplication**합니다.
- 결과: `PLA_6_13_hifi_unique_sequences.csv.gz` (`sequence,read_count,length_bp`), 별도 summary.
- 원본 BAM 122,715,338,770 bytes. **다운로드 파일로 보존하지 않고** Drive API에서 스트리밍하여 `/content` 임시 sort 파일로 처리합니다.
- 주의: 122.7GB의 전체 전송은 필요하며, Colab 세션 한도·네트워크 제한·Drive quota에 의해 중단될 수 있습니다. 중단 시 전체 스트림 재시작이 필요합니다.
- 프라이머/인덱스 trimming 및 strand canonicalization은 적용하지 않습니다. 기존 Illumina exact-sequence 표와 동일한 3개 열이지만 library/read 구조가 다르므로 counts를 직접 같은 의미로 해석해서는 안 됩니다.


In [ ]:
!which samtools || (apt-get update -qq >/dev/null && apt-get install -y -qq samtools >/dev/null)
!samtools --version | head -1


In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
import google.auth
from google.auth.transport.requests import AuthorizedSession
import subprocess, threading, time, gzip, csv, shutil, os, json
from pathlib import Path
from urllib.parse import quote

BAM_FILE_ID = '1RthcVb15hJpLF3V_P9vtgDCE1q9yH-Ln'
EXPECTED_BAM_BYTES = 122715338770
OUT = Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PacBio_PLA_6_13')
TMP = Path('/content/pacbio_pla613_tmp')
OUT.mkdir(parents=True,exist_ok=True)
TMP.mkdir(parents=True,exist_ok=True)
RESULT=OUT/'PLA_6_13_hifi_unique_sequences.csv.gz'
SUMMARY=OUT/'PLA_6_13_hifi_unique_summary.json'
SORT_RAM='2G'
SORT_THREADS=4
FORCE_RECOMPUTE=False
print('Output:',OUT)


## Run

`samtools fastq`는 BAM record를 FASTQ로 변환하며, `awk`는 sequence 줄만 추출합니다. `sort | uniq -c`는 **정확히 동일한 sequence만 묶습니다.** 전체 122.7GB BAM은 임시 파일로 다운로드하지 않고 청크 단위로 파이프에 전달합니다. 정렬 임시 파일 저장 공간은 별도로 필요합니다.


In [ ]:
def verify_csv(path):
    unique = total = bad = 0
    with gzip.open(path,'rt') as handle:
        reader=csv.DictReader(handle)
        assert reader.fieldnames==['sequence','read_count','length_bp'], reader.fieldnames
        for row in reader:
            s=row['sequence']; n=int(row['read_count']); length=int(row['length_bp'])
            if not s or length != len(s) or n <= 0: bad+=1
            total+=n; unique+=1
    if bad: raise RuntimeError(f'{bad} invalid rows')
    return dict(unique_sequences=unique,total_hifi_reads=total)

if RESULT.exists() and not FORCE_RECOMPUTE:
    print('Existing file:',RESULT)
    print('Verified:',verify_csv(RESULT))
else:
    # End-to-end pipefail: Drive HTTP stream -> samtools -> awk -> sort -> uniq -> gzip
    creds,_ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
    session=AuthorizedSession(creds)
    media_url='https://www.googleapis.com/drive/v3/files/'+quote(BAM_FILE_ID,safe='')+'?alt=media&supportsAllDrives=true'
    output_tmp=TMP/'PLA_6_13_hifi_unique_sequences.csv.gz.partial'
    sort_dir=TMP/'sort_tmp'
    sort_dir.mkdir(exist_ok=True)
    bash_script=r'''set -o pipefail
samtools fastq -@ 4 - 2> "$3" | awk 'NR%4==2 {print}' | LC_ALL=C sort --parallel=4 -S "$4" -T "$5" | LC_ALL=C uniq -c | awk 'BEGIN{print "sequence,read_count,length_bp"} {n=$1;$1="";sub(/^ /,"");print $0 "," n "," length($0)}' | gzip -1 > "$2"
'''
    subprocess.run(['bash','-n','-c',bash_script],check=True)
    start=time.monotonic()
    with open(TMP/'samtools.log','w') as log:
        proc=subprocess.Popen(['bash','-c',bash_script,'bash','-',str(output_tmp),
                               str(TMP/'samtools_stderr.log'),SORT_RAM,str(sort_dir)],
                               stdin=subprocess.PIPE,stderr=log)
        nbytes=0
        try:
            # A single request sends the BAM body in chunks. No local BAM file is created.
            with session.get(media_url,stream=True,timeout=(60,600)) as resp:
                resp.raise_for_status()
                content_length=int(resp.headers.get('Content-Length','0'))
                if content_length and content_length!=EXPECTED_BAM_BYTES:
                    raise RuntimeError(f'Unexpected content length: {content_length}')
                last_progress=start
                for chunk in resp.iter_content(chunk_size=8*1024*1024):
                    if not chunk: continue
                    proc.stdin.write(chunk)
                    nbytes+=len(chunk)
                    now=time.monotonic()
                    if now-last_progress>60:
                        print(f'Streamed {nbytes/1e9:.2f}/{EXPECTED_BAM_BYTES/1e9:.2f} GB; '
                              f'elapsed {(now-start)/60:.1f} min',flush=True)
                        last_progress=now
            proc.stdin.close()
            rc=proc.wait()
            if nbytes!=EXPECTED_BAM_BYTES:
                raise RuntimeError(f'BAM byte count incomplete: {nbytes} != {EXPECTED_BAM_BYTES}')
            if rc:
                details=(TMP/'samtools_stderr.log').read_text()[-3000:]
                raise RuntimeError(f'Pipeline exit={rc}: {details}')
            stats=verify_csv(output_tmp)
            shutil.copyfile(output_tmp,RESULT.with_suffix('.gz.partial'))
            verify_csv(RESULT.with_suffix('.gz.partial'))
            RESULT.with_suffix('.gz.partial').replace(RESULT)
            summary=dict(sample='PLA_6_13',source='PacBio HiFi BAM',bam_bytes=nbytes,
                         method='samtools fastq | exact sequence sort uniq',
                         elapsed_seconds=round(time.monotonic()-start,1),**stats)
            SUMMARY.write_text(json.dumps(summary,indent=2))
            print('DONE',summary)
        except BaseException:
            try: proc.stdin.close()
            except Exception: pass
            if proc.poll() is None: proc.terminate()
            proc.wait()
            raise
        finally:
            output_tmp.unlink(missing_ok=True)


## 결과 확인

`PLA_6_13_hifi_unique_sequences.csv.gz`는 read count 내림차순이 아니라 **서열순**입니다. 필요하면 추후 별도 정렬하세요. BAM의 reverse-strand FLAG에 따른 FASTQ 방향 처리 등은 `samtools fastq` 동작에 따릅니다.

**추가 검증 권장:** 정렬/매핑 여부 확인, HiFi read 길이 분포, PCR primer 위치, read 방향 통일, primer/index 제거 후 별도 count. 현재 결과는 raw HiFi read exact-count입니다.
